# **Lucas–Kanade - OpenCV**

En esta sección, implementamos el método de Lucas-Kanade usando el código optimizado proporcionado por [OpenCV](https://docs.opencv.org/3.4/d4/dee/tutorial_optical_flow.html).

In [20]:
import numpy as np
import cv2 as cv

VIDEO_PATH = "videos/people2.mp4"

cap = cv.VideoCapture(VIDEO_PATH)

# Parámetros para detectar esquinas (ShiTomasi)
feature_params = dict( maxCorners = 100,
                       qualityLevel = 0.3,
                       minDistance = 7,
                       blockSize = 7 )
# Parámetros de Lucas-Kanade
lk_params = dict( winSize  = (15, 15),
                  maxLevel = 2,
                  criteria = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03))
# Crear colores aleatorios
color = np.random.randint(0, 255, (100, 3))
# Primer frame
ret, old_frame = cap.read()
old_gray = cv.cvtColor(old_frame, cv.COLOR_BGR2GRAY)
# Esquinas en el primer frame
p0 = cv.goodFeaturesToTrack(old_gray, mask = None, **feature_params)
# Crear una imagen de máscara para dibujar
mask = np.zeros_like(old_frame)
while(1):
    ret, frame = cap.read()
    if not ret:
        break
    frame_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    # Calculamos flujo óptico (Lucas-Kanade)
    p1, st, err = cv.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)
    # seleccionamos puntos válidos
    if p1 is not None:
        good_new = p1[st==1]
        good_old = p0[st==1]
    # Dibujamos las trayectorias
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel()
        c, d = old.ravel()
        mask = cv.line(mask, (int(a), int(b)), (int(c), int(d)), color[i].tolist(), 2)
        frame = cv.circle(frame, (int(a), int(b)), 5, color[i].tolist(), -1)
    img = cv.add(frame, mask)
    cv.imshow('Lucas-Kanade (disperso)', img)
    k = cv.waitKey(30) & 0xff
    if k == 27:
        break
    # Actualizamos el frame y los puntos
    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)
cv.destroyAllWindows()